In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from tqdm import tqdm


# ============================================================
# 1. Device
# ============================================================

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


# ============================================================
# 2. Transforms
#    Convert MNIST to RGB 32x32 so both datasets have same shape
# ============================================================

mnist_train_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5, 0.5, 0.5),
        std=(0.5, 0.5, 0.5)
    )
])

mnist_test_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5, 0.5, 0.5),
        std=(0.5, 0.5, 0.5)
    )
])

cifar_train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5071, 0.4867, 0.4408),
        std=(0.2675, 0.2565, 0.2761)
    )
])

cifar_test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5071, 0.4867, 0.4408),
        std=(0.2675, 0.2565, 0.2761)
    )
])


# ============================================================
# 3. Dataset Wrappers
# ============================================================

class MNISTProxyVLMDataset(Dataset):
    def __init__(self, train=True, root="./data"):
        self.dataset = datasets.MNIST(
            root=root,
            train=train,
            download=True,
            transform=mnist_train_transform if train else mnist_test_transform
        )

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, label = self.dataset[idx]

        # MNIST labels stay 0-9
        unified_label = label

        return image, unified_label


class CIFAR100ProxyVLMDataset(Dataset):
    def __init__(self, train=True, root="./data"):
        self.dataset = datasets.CIFAR100(
            root=root,
            train=train,
            download=True,
            transform=cifar_train_transform if train else cifar_test_transform
        )

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, label = self.dataset[idx]

        # CIFAR-100 labels are shifted by 10
        # MNIST: 0-9
        # CIFAR-100: 10-109
        unified_label = label + 10

        return image, unified_label


# ============================================================
# 4. Load Datasets
# ============================================================

mnist_train = MNISTProxyVLMDataset(train=True)
mnist_test = MNISTProxyVLMDataset(train=False)

cifar_train = CIFAR100ProxyVLMDataset(train=True)
cifar_test = CIFAR100ProxyVLMDataset(train=False)

train_dataset = ConcatDataset([mnist_train, cifar_train])
test_dataset = ConcatDataset([mnist_test, cifar_test])

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=2,
    pin_memory=True if device == "cuda" else False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=2,
    pin_memory=True if device == "cuda" else False
)


# ============================================================
# 5. Build Unified Class Prompts
# ============================================================

cifar_class_names = datasets.CIFAR100(
    root="./data",
    train=True,
    download=False
).classes

mnist_prompts = [
    f"a photo of digit {i}"
    for i in range(10)
]

cifar_prompts = [
    f"a photo of a {name.replace('_', ' ')}"
    for name in cifar_class_names
]

all_prompts = mnist_prompts + cifar_prompts

all_class_names = (
    [f"digit_{i}" for i in range(10)]
    + cifar_class_names
)

num_classes = len(all_prompts)

print("Total classes:", num_classes)
print("First 10 prompts:", all_prompts[:10])
print("Example CIFAR prompts:", all_prompts[10:20])


# ============================================================
# 6. Text Processor
# ============================================================

class ProxyVLMTextProcessor:
    def __init__(self, prompts):
        self.prompts = prompts

        vocab = {
            "<pad>": 0,
            "<unk>": 1
        }

        for prompt in prompts:
            for word in prompt.lower().split():
                if word not in vocab:
                    vocab[word] = len(vocab)

        self.token_to_id = vocab
        self.id_to_token = {v: k for k, v in vocab.items()}
        self.vocab_size = len(vocab)
        self.max_len = max(len(prompt.split()) for prompt in prompts)

    def encode_prompt(self, prompt):
        words = prompt.lower().split()

        ids = [
            self.token_to_id.get(word, self.token_to_id["<unk>"])
            for word in words
        ]

        while len(ids) < self.max_len:
            ids.append(self.token_to_id["<pad>"])

        return torch.tensor(ids, dtype=torch.long)

    def build_all_class_token_ids(self):
        encoded = [
            self.encode_prompt(prompt)
            for prompt in self.prompts
        ]

        return torch.stack(encoded, dim=0)


text_processor = ProxyVLMTextProcessor(all_prompts)

class_token_ids = text_processor.build_all_class_token_ids().to(device)

print("Vocabulary size:", text_processor.vocab_size)
print("Max prompt length:", text_processor.max_len)


# ============================================================
# 7. Shared Image Encoder
# ============================================================

class SharedImageEncoder(nn.Module):
    def __init__(self, embed_dim=128):
        super().__init__()

        self.features = nn.Sequential(
            # Input: 3 x 32 x 32
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 32 x 16 x 16

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 64 x 8 x 8

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 128 x 4 x 4
        )

        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, embed_dim)
        )

    def forward(self, images):
        x = self.features(images)
        x = self.proj(x)
        x = F.normalize(x, dim=-1)
        return x


# ============================================================
# 8. Tiny Text Encoder
# ============================================================

class TinyTextEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, text_hidden_dim=128, pad_id=0):
        super().__init__()

        self.pad_id = pad_id

        self.token_embedding = nn.Embedding(
            vocab_size,
            text_hidden_dim,
            padding_idx=pad_id
        )

        self.proj = nn.Sequential(
            nn.Linear(text_hidden_dim, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, embed_dim)
        )

    def forward(self, token_ids):
        x = self.token_embedding(token_ids)

        mask = (token_ids != self.pad_id).unsqueeze(-1).float()
        x = x * mask

        summed = x.sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1.0)

        x = summed / counts

        x = self.proj(x)
        x = F.normalize(x, dim=-1)

        return x


# ============================================================
# 9. Proxy VLM Model
# ============================================================

class MultiDatasetProxyVLM(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, text_hidden_dim=128, pad_id=0):
        super().__init__()

        self.image_encoder = SharedImageEncoder(embed_dim=embed_dim)

        self.text_encoder = TinyTextEncoder(
            vocab_size=vocab_size,
            embed_dim=embed_dim,
            text_hidden_dim=text_hidden_dim,
            pad_id=pad_id
        )

        self.logit_scale = nn.Parameter(torch.tensor(1.0))

    def encode_image(self, images):
        return self.image_encoder(images)

    def encode_text(self, token_ids):
        return self.text_encoder(token_ids)

    def forward(self, images, class_token_ids):
        image_features = self.encode_image(images)
        text_features = self.encode_text(class_token_ids)

        scale = self.logit_scale.exp().clamp(max=100)

        logits = scale * image_features @ text_features.T

        return logits


# ============================================================
# 10. Initialize
# ============================================================

model = MultiDatasetProxyVLM(
    vocab_size=text_processor.vocab_size,
    embed_dim=128,
    text_hidden_dim=128,
    pad_id=text_processor.token_to_id["<pad>"]
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

criterion = nn.CrossEntropyLoss()


# ============================================================
# 11. Training
# ============================================================

def train_one_epoch(model, loader, optimizer, criterion, class_token_ids):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    mnist_correct = 0
    mnist_total = 0

    cifar_correct = 0
    cifar_total = 0

    loop = tqdm(loader, desc="Training", leave=False)

    for images, labels in loop:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(images, class_token_ids)

        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        preds = logits.argmax(dim=1)

        total_loss += loss.item() * labels.size(0)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

        mnist_mask = labels < 10
        cifar_mask = labels >= 10

        if mnist_mask.any():
            mnist_correct += (preds[mnist_mask] == labels[mnist_mask]).sum().item()
            mnist_total += mnist_mask.sum().item()

        if cifar_mask.any():
            cifar_correct += (preds[cifar_mask] == labels[cifar_mask]).sum().item()
            cifar_total += cifar_mask.sum().item()

        loop.set_postfix(
            loss=loss.item(),
            acc=100 * total_correct / total_samples
        )

    avg_loss = total_loss / total_samples
    total_acc = 100 * total_correct / total_samples
    mnist_acc = 100 * mnist_correct / mnist_total if mnist_total > 0 else 0
    cifar_acc = 100 * cifar_correct / cifar_total if cifar_total > 0 else 0

    return avg_loss, total_acc, mnist_acc, cifar_acc


# ============================================================
# 12. Evaluation
# ============================================================

def evaluate(model, loader, criterion, class_token_ids):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    mnist_correct = 0
    mnist_total = 0

    cifar_correct = 0
    cifar_total = 0

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Evaluating", leave=False):
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images, class_token_ids)
            loss = criterion(logits, labels)

            preds = logits.argmax(dim=1)

            total_loss += loss.item() * labels.size(0)
            total_correct += (preds == labels).sum().item()
            total_samples += labels.size(0)

            mnist_mask = labels < 10
            cifar_mask = labels >= 10

            if mnist_mask.any():
                mnist_correct += (preds[mnist_mask] == labels[mnist_mask]).sum().item()
                mnist_total += mnist_mask.sum().item()

            if cifar_mask.any():
                cifar_correct += (preds[cifar_mask] == labels[cifar_mask]).sum().item()
                cifar_total += cifar_mask.sum().item()

    avg_loss = total_loss / total_samples
    total_acc = 100 * total_correct / total_samples
    mnist_acc = 100 * mnist_correct / mnist_total if mnist_total > 0 else 0
    cifar_acc = 100 * cifar_correct / cifar_total if cifar_total > 0 else 0

    return avg_loss, total_acc, mnist_acc, cifar_acc


# ============================================================
# 13. Main Training Loop
# ============================================================

num_epochs = 20
best_total_acc = 0.0

for epoch in range(num_epochs):
    print(f"\nEpoch [{epoch + 1}/{num_epochs}]")

    train_loss, train_acc, train_mnist_acc, train_cifar_acc = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        class_token_ids
    )

    test_loss, test_acc, test_mnist_acc, test_cifar_acc = evaluate(
        model,
        test_loader,
        criterion,
        class_token_ids
    )

    print(
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"MNIST: {train_mnist_acc:.2f}% | "
        f"CIFAR-100: {train_cifar_acc:.2f}%"
    )

    print(
        f"Test Loss: {test_loss:.4f} | "
        f"Test Acc: {test_acc:.2f}% | "
        f"MNIST: {test_mnist_acc:.2f}% | "
        f"CIFAR-100: {test_cifar_acc:.2f}%"
    )

    if test_acc > best_total_acc:
        best_total_acc = test_acc

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "all_prompts": all_prompts,
                "all_class_names": all_class_names,
                "token_to_id": text_processor.token_to_id,
                "best_total_acc": best_total_acc,
                "test_mnist_acc": test_mnist_acc,
                "test_cifar_acc": test_cifar_acc,
            },
            "mnist_cifar100_proxy_vlm.pth"
        )

        print(f"Saved best model. Best total acc: {best_total_acc:.2f}%")


print("\nTraining complete.")
print(f"Best Total Test Accuracy: {best_total_acc:.2f}%")

Using device: cuda


100%|██████████| 9.91M/9.91M [00:00<00:00, 17.1MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.08MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 9.41MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 1.44MB/s]
100%|██████████| 169M/169M [02:43<00:00, 1.03MB/s] 


Total classes: 110
First 10 prompts: ['a photo of digit 0', 'a photo of digit 1', 'a photo of digit 2', 'a photo of digit 3', 'a photo of digit 4', 'a photo of digit 5', 'a photo of digit 6', 'a photo of digit 7', 'a photo of digit 8', 'a photo of digit 9']
Example CIFAR prompts: ['a photo of a apple', 'a photo of a aquarium fish', 'a photo of a baby', 'a photo of a bear', 'a photo of a beaver', 'a photo of a bed', 'a photo of a bee', 'a photo of a beetle', 'a photo of a bicycle', 'a photo of a bottle']
Vocabulary size: 121
Max prompt length: 6

Epoch [1/20]


Training:   0%|          | 0/860 [00:00<?, ?it/s]

In [ ]:
import matplotlib.pyplot as plt

model.eval()

idx = 0
image, label = test_dataset[idx]

with torch.no_grad():
    input_image = image.unsqueeze(0).to(device)
    logits = model(input_image, class_token_ids)
    pred = logits.argmax(dim=1).item()

print("True label:", all_class_names[label])
print("Predicted label:", all_class_names[pred])
print("True prompt:", all_prompts[label])
print("Predicted prompt:", all_prompts[pred])

# Display normalized image approximately
img = image.cpu().permute(1, 2, 0)

# simple visualization clamp
img = (img - img.min()) / (img.max() - img.min() + 1e-8)

plt.imshow(img)
plt.title(f"True: {all_class_names[label]} | Pred: {all_class_names[pred]}")
plt.axis("off")
plt.show()

# For the training of the model

In [ ]:
# ============================================================
# 7. Text Processor
# ============================================================

class ProxyVLMTextProcessor:
    def __init__(self, prompts):
        self.prompts = prompts

        vocab = {
            "<pad>": 0,
            "<unk>": 1
        }

        for prompt in prompts:
            words = prompt.lower().replace("-", " ").replace("_", " ").split()

            for word in words:
                if word not in vocab:
                    vocab[word] = len(vocab)

        self.token_to_id = vocab
        self.id_to_token = {v: k for k, v in vocab.items()}
        self.vocab_size = len(vocab)

        self.max_len = max(
            len(prompt.lower().replace("-", " ").replace("_", " ").split())
            for prompt in prompts
        )

    def encode_prompt(self, prompt):
        words = prompt.lower().replace("-", " ").replace("_", " ").split()

        ids = [
            self.token_to_id.get(word, self.token_to_id["<unk>"])
            for word in words
        ]

        while len(ids) < self.max_len:
            ids.append(self.token_to_id["<pad>"])

        return torch.tensor(ids, dtype=torch.long)

    def build_all_class_token_ids(self):
        encoded = [
            self.encode_prompt(prompt)
            for prompt in self.prompts
        ]

        return torch.stack(encoded, dim=0)


text_processor = ProxyVLMTextProcessor(all_prompts)
class_token_ids = text_processor.build_all_class_token_ids().to(device)

print("Vocabulary size:", text_processor.vocab_size)
print("Max prompt length:", text_processor.max_len)


# ============================================================
# 8. Model
# ============================================================

class SharedImageEncoder(nn.Module):
    def __init__(self, embed_dim=256):
        super().__init__()

        self.features = nn.Sequential(
            # Input: 3 x 64 x 64

            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 32 x 32 x 32

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 64 x 16 x 16

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 128 x 8 x 8

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),

            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(512, embed_dim)
        )

    def forward(self, images):
        x = self.features(images)
        x = self.proj(x)
        x = F.normalize(x, dim=-1)
        return x


class TinyTextEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, text_hidden_dim=256, pad_id=0):
        super().__init__()

        self.pad_id = pad_id

        self.token_embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=text_hidden_dim,
            padding_idx=pad_id
        )

        self.proj = nn.Sequential(
            nn.Linear(text_hidden_dim, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, embed_dim)
        )

    def forward(self, token_ids):
        x = self.token_embedding(token_ids)

        mask = (token_ids != self.pad_id).unsqueeze(-1).float()
        x = x * mask

        summed = x.sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1.0)

        x = summed / counts

        x = self.proj(x)
        x = F.normalize(x, dim=-1)

        return x


class MultiDatasetProxyVLM(nn.Module):
    def __init__(
        self,
        vocab_size,
        embed_dim=256,
        text_hidden_dim=256,
        pad_id=0
    ):
        super().__init__()

        self.image_encoder = SharedImageEncoder(embed_dim=embed_dim)

        self.text_encoder = TinyTextEncoder(
            vocab_size=vocab_size,
            embed_dim=embed_dim,
            text_hidden_dim=text_hidden_dim,
            pad_id=pad_id
        )

        self.logit_scale = nn.Parameter(torch.tensor(1.0))

    def encode_image(self, images):
        return self.image_encoder(images)

    def encode_text(self, token_ids):
        return self.text_encoder(token_ids)

    def forward(self, images, class_token_ids):
        image_features = self.encode_image(images)
        text_features = self.encode_text(class_token_ids)

        scale = self.logit_scale.exp().clamp(max=100)

        logits = scale * image_features @ text_features.T

        return logits

In [ ]:
# ============================================================
# Complete Proxy VLM Training Code
# MNIST + CIFAR-100 + TinyImageNet-200
# ============================================================

import os
import urllib.request
import zipfile
import shutil

import torch
import torch.nn as nn
import torch.nn.functional as F

from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from tqdm import tqdm


# ============================================================
# 1. Device
# ============================================================

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


# ============================================================
# 2. TinyImageNet Download and Preparation
# ============================================================

def download_tiny_imagenet(root="./data"):
    os.makedirs(root, exist_ok=True)

    dataset_dir = os.path.join(root, "tiny-imagenet-200")
    zip_path = os.path.join(root, "tiny-imagenet-200.zip")

    if os.path.exists(dataset_dir):
        print("TinyImageNet already exists.")
        return dataset_dir

    url = "http://cs231n.stanford.edu/tiny-imagenet-200.zip"

    print("Downloading TinyImageNet-200...")
    urllib.request.urlretrieve(url, zip_path)

    print("Extracting TinyImageNet-200...")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(root)

    print("TinyImageNet downloaded and extracted.")
    return dataset_dir


def prepare_tiny_imagenet_val(dataset_dir):
    val_dir = os.path.join(dataset_dir, "val")
    val_images_dir = os.path.join(val_dir, "images")
    annotations_file = os.path.join(val_dir, "val_annotations.txt")
    prepared_marker = os.path.join(val_dir, ".prepared")

    if os.path.exists(prepared_marker):
        print("TinyImageNet validation folder already prepared.")
        return

    if not os.path.exists(val_images_dir):
        print("TinyImageNet validation folder already reorganized.")
        with open(prepared_marker, "w") as f:
            f.write("prepared")
        return

    print("Preparing TinyImageNet validation folder...")

    with open(annotations_file, "r") as f:
        lines = f.readlines()

    for line in lines:
        parts = line.strip().split("\t")
        img_name = parts[0]
        class_id = parts[1]

        class_dir = os.path.join(val_dir, class_id)
        os.makedirs(class_dir, exist_ok=True)

        src = os.path.join(val_images_dir, img_name)
        dst = os.path.join(class_dir, img_name)

        if os.path.exists(src):
            shutil.move(src, dst)

    shutil.rmtree(val_images_dir)

    with open(prepared_marker, "w") as f:
        f.write("prepared")

    print("TinyImageNet validation folder prepared.")


def load_tiny_imagenet_class_names(dataset_dir):
    words_file = os.path.join(dataset_dir, "words.txt")
    wnids_file = os.path.join(dataset_dir, "wnids.txt")

    wnid_to_words = {}

    with open(words_file, "r") as f:
        for line in f:
            parts = line.strip().split("\t")
            wnid = parts[0]
            words = parts[1]
            wnid_to_words[wnid] = words

    with open(wnids_file, "r") as f:
        wnids = [line.strip() for line in f.readlines()]

    class_names = []

    for wnid in wnids:
        readable = wnid_to_words[wnid].split(",")[0]
        readable = readable.replace("_", " ")
        class_names.append(readable)

    return wnids, class_names


# ============================================================
# 3. Image Transforms
# ============================================================

IMG_SIZE = 64

mnist_train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5, 0.5, 0.5),
        std=(0.5, 0.5, 0.5)
    )
])

mnist_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5, 0.5, 0.5),
        std=(0.5, 0.5, 0.5)
    )
])

cifar_train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(IMG_SIZE, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5071, 0.4867, 0.4408),
        std=(0.2675, 0.2565, 0.2761)
    )
])

cifar_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5071, 0.4867, 0.4408),
        std=(0.2675, 0.2565, 0.2761)
    )
])

tiny_train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4802, 0.4481, 0.3975),
        std=(0.2302, 0.2265, 0.2262)
    )
])

tiny_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4802, 0.4481, 0.3975),
        std=(0.2302, 0.2265, 0.2262)
    )
])


# ============================================================
# 4. Dataset Wrappers
# ============================================================

class MNISTProxyVLMDataset(Dataset):
    def __init__(self, train=True, root="./data"):
        self.dataset = datasets.MNIST(
            root=root,
            train=train,
            download=True,
            transform=mnist_train_transform if train else mnist_test_transform
        )

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, label = self.dataset[idx]

        # Unified label range:
        # MNIST: 0-9
        unified_label = label
        domain_id = 0

        return image, unified_label, domain_id


class CIFAR100ProxyVLMDataset(Dataset):
    def __init__(self, train=True, root="./data"):
        self.dataset = datasets.CIFAR100(
            root=root,
            train=train,
            download=True,
            transform=cifar_train_transform if train else cifar_test_transform
        )

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, label = self.dataset[idx]

        # Unified label range:
        # CIFAR-100: 10-109
        unified_label = label + 10
        domain_id = 1

        return image, unified_label, domain_id


class TinyImageNetProxyVLMDataset(Dataset):
    def __init__(self, train=True, root="./data"):
        dataset_dir = download_tiny_imagenet(root=root)
        prepare_tiny_imagenet_val(dataset_dir)

        split_dir = os.path.join(dataset_dir, "train" if train else "val")

        self.dataset = datasets.ImageFolder(
            root=split_dir,
            transform=tiny_train_transform if train else tiny_test_transform
        )

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, label = self.dataset[idx]

        # Unified label range:
        # TinyImageNet-200: 110-309
        unified_label = label + 110
        domain_id = 2

        return image, unified_label, domain_id


# ============================================================
# 5. Load Datasets
# ============================================================

mnist_train = MNISTProxyVLMDataset(train=True)
mnist_test = MNISTProxyVLMDataset(train=False)

cifar_train = CIFAR100ProxyVLMDataset(train=True)
cifar_test = CIFAR100ProxyVLMDataset(train=False)

tiny_train = TinyImageNetProxyVLMDataset(train=True)
tiny_test = TinyImageNetProxyVLMDataset(train=False)

train_dataset = ConcatDataset([
    mnist_train,
    cifar_train,
    tiny_train
])

test_dataset = ConcatDataset([
    mnist_test,
    cifar_test,
    tiny_test
])

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=2,
    pin_memory=True if device == "cuda" else False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=2,
    pin_memory=True if device == "cuda" else False
)


# ============================================================
# 6. Build Unified Text Prompts
# ============================================================

cifar_class_names = datasets.CIFAR100(
    root="./data",
    train=True,
    download=False
).classes

tiny_dataset_dir = os.path.join("./data", "tiny-imagenet-200")
tiny_wnids, tiny_class_names = load_tiny_imagenet_class_names(tiny_dataset_dir)

mnist_prompts = [
    f"a photo of digit {i}"
    for i in range(10)
]

cifar_prompts = [
    f"a photo of a {name.replace('_', ' ')}"
    for name in cifar_class_names
]

tiny_prompts = [
    f"a photo of a {name}"
    for name in tiny_class_names
]

all_prompts = mnist_prompts + cifar_prompts + tiny_prompts

all_class_names = (
    [f"mnist_digit_{i}" for i in range(10)]
    + [f"cifar100_{name}" for name in cifar_class_names]
    + [f"tinyimagenet_{name}" for name in tiny_class_names]
)

num_classes = len(all_prompts)

print("Total classes:", num_classes)
print("MNIST label range: 0-9")
print("CIFAR-100 label range: 10-109")
print("TinyImageNet label range: 110-309")
print("Example prompts:", all_prompts[:5])





model = MultiDatasetProxyVLM(
    vocab_size=text_processor.vocab_size,
    embed_dim=256,
    text_hidden_dim=256,
    pad_id=text_processor.token_to_id["<pad>"]
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

criterion = nn.CrossEntropyLoss()


# ============================================================
# 9. Training Function
# ============================================================

def train_one_epoch(model, loader, optimizer, criterion, class_token_ids, device):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    mnist_correct, mnist_total = 0, 0
    cifar_correct, cifar_total = 0, 0
    tiny_correct, tiny_total = 0, 0

    loop = tqdm(loader, desc="Training", leave=False)

    for images, labels, domain_ids in loop:
        images = images.to(device)
        labels = labels.to(device)
        domain_ids = domain_ids.to(device)

        optimizer.zero_grad()

        logits = model(images, class_token_ids)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        preds = logits.argmax(dim=1)

        total_loss += loss.item() * labels.size(0)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

        mnist_mask = domain_ids == 0
        cifar_mask = domain_ids == 1
        tiny_mask = domain_ids == 2

        if mnist_mask.any():
            mnist_correct += (preds[mnist_mask] == labels[mnist_mask]).sum().item()
            mnist_total += mnist_mask.sum().item()

        if cifar_mask.any():
            cifar_correct += (preds[cifar_mask] == labels[cifar_mask]).sum().item()
            cifar_total += cifar_mask.sum().item()

        if tiny_mask.any():
            tiny_correct += (preds[tiny_mask] == labels[tiny_mask]).sum().item()
            tiny_total += tiny_mask.sum().item()

        loop.set_postfix(
            loss=f"{loss.item():.4f}",
            acc=f"{100 * total_correct / total_samples:.2f}%"
        )

    avg_loss = total_loss / total_samples
    total_acc = 100 * total_correct / total_samples

    mnist_acc = 100 * mnist_correct / mnist_total if mnist_total > 0 else 0
    cifar_acc = 100 * cifar_correct / cifar_total if cifar_total > 0 else 0
    tiny_acc = 100 * tiny_correct / tiny_total if tiny_total > 0 else 0

    return avg_loss, total_acc, mnist_acc, cifar_acc, tiny_acc


# ============================================================
# 10. Evaluation Function
# ============================================================

def evaluate(model, loader, criterion, class_token_ids, device):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    mnist_correct, mnist_total = 0, 0
    cifar_correct, cifar_total = 0, 0
    tiny_correct, tiny_total = 0, 0

    with torch.no_grad():
        for images, labels, domain_ids in tqdm(loader, desc="Evaluating", leave=False):
            images = images.to(device)
            labels = labels.to(device)
            domain_ids = domain_ids.to(device)

            logits = model(images, class_token_ids)
            loss = criterion(logits, labels)

            preds = logits.argmax(dim=1)

            total_loss += loss.item() * labels.size(0)
            total_correct += (preds == labels).sum().item()
            total_samples += labels.size(0)

            mnist_mask = domain_ids == 0
            cifar_mask = domain_ids == 1
            tiny_mask = domain_ids == 2

            if mnist_mask.any():
                mnist_correct += (preds[mnist_mask] == labels[mnist_mask]).sum().item()
                mnist_total += mnist_mask.sum().item()

            if cifar_mask.any():
                cifar_correct += (preds[cifar_mask] == labels[cifar_mask]).sum().item()
                cifar_total += cifar_mask.sum().item()

            if tiny_mask.any():
                tiny_correct += (preds[tiny_mask] == labels[tiny_mask]).sum().item()
                tiny_total += tiny_mask.sum().item()

    avg_loss = total_loss / total_samples
    total_acc = 100 * total_correct / total_samples

    mnist_acc = 100 * mnist_correct / mnist_total if mnist_total > 0 else 0
    cifar_acc = 100 * cifar_correct / cifar_total if cifar_total > 0 else 0
    tiny_acc = 100 * tiny_correct / tiny_total if tiny_total > 0 else 0

    return avg_loss, total_acc, mnist_acc, cifar_acc, tiny_acc


# ============================================================
# 11. Train 10 Epochs and Save Model
# ============================================================

num_epochs = 10

best_total_acc = 0.0

best_model_path = "best_proxy_vlm_mnist_cifar100_tinyimagenet.pth"
final_model_path = "final_proxy_vlm_mnist_cifar100_tinyimagenet.pth"

for epoch in range(num_epochs):
    print(f"\nEpoch [{epoch + 1}/{num_epochs}]")

    train_loss, train_acc, train_mnist_acc, train_cifar_acc, train_tiny_acc = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        class_token_ids=class_token_ids,
        device=device
    )

    test_loss, test_acc, test_mnist_acc, test_cifar_acc, test_tiny_acc = evaluate(
        model=model,
        loader=test_loader,
        criterion=criterion,
        class_token_ids=class_token_ids,
        device=device
    )

    print(
        f"Train Loss: {train_loss:.4f} | "
        f"Total Acc: {train_acc:.2f}% | "
        f"MNIST: {train_mnist_acc:.2f}% | "
        f"CIFAR-100: {train_cifar_acc:.2f}% | "
        f"TinyImageNet: {train_tiny_acc:.2f}%"
    )

    print(
        f"Test Loss: {test_loss:.4f} | "
        f"Total Acc: {test_acc:.2f}% | "
        f"MNIST: {test_mnist_acc:.2f}% | "
        f"CIFAR-100: {test_cifar_acc:.2f}% | "
        f"TinyImageNet: {test_tiny_acc:.2f}%"
    )

    if test_acc > best_total_acc:
        best_total_acc = test_acc

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),

                "best_total_acc": best_total_acc,
                "test_total_acc": test_acc,
                "test_mnist_acc": test_mnist_acc,
                "test_cifar_acc": test_cifar_acc,
                "test_tiny_acc": test_tiny_acc,

                "all_prompts": all_prompts,
                "all_class_names": all_class_names,
                "token_to_id": text_processor.token_to_id,
                "id_to_token": text_processor.id_to_token,
                "vocab_size": text_processor.vocab_size,
                "max_len": text_processor.max_len,
                "class_token_ids": class_token_ids.cpu(),

                "embed_dim": 256,
                "text_hidden_dim": 256,
                "num_classes": len(all_prompts),
            },
            best_model_path
        )

        print(f"Best model saved: {best_model_path}")


torch.save(
    {
        "epoch": num_epochs,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),

        "final_total_acc": test_acc,
        "final_mnist_acc": test_mnist_acc,
        "final_cifar_acc": test_cifar_acc,
        "final_tiny_acc": test_tiny_acc,

        "all_prompts": all_prompts,
        "all_class_names": all_class_names,
        "token_to_id": text_processor.token_to_id,
        "id_to_token": text_processor.id_to_token,
        "vocab_size": text_processor.vocab_size,
        "max_len": text_processor.max_len,
        "class_token_ids": class_token_ids.cpu(),

        "embed_dim": 256,
        "text_hidden_dim": 256,
        "num_classes": len(all_prompts),
    },
    final_model_path
)

print("\nTraining complete.")
print(f"Best Total Test Accuracy: {best_total_acc:.2f}%")
print(f"Best model saved to: {best_model_path}")
print(f"Final model saved to: {final_model_path}")

Using device: cuda
TinyImageNet already exists.
TinyImageNet validation folder already prepared.
TinyImageNet already exists.
TinyImageNet validation folder already prepared.
Total classes: 310
MNIST label range: 0-9
CIFAR-100 label range: 10-109
TinyImageNet label range: 110-309
Example prompts: ['a photo of digit 0', 'a photo of digit 1', 'a photo of digit 2', 'a photo of digit 3', 'a photo of digit 4']
Vocabulary size: 356
Max prompt length: 7

Epoch [1/10]


Training:   0%|          | 0/1641 [00:00<?, ?it/s]

# Test dataset

In [1]:
# ============================================================
# Test Saved Proxy VLM on MNIST, CIFAR-100, TinyImageNet-200
# ============================================================

import os
import torch
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm


# ============================================================
# 1. Device
# ============================================================

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


# ============================================================
# 2. Load Saved Model
# ============================================================

checkpoint_path = "best_proxy_vlm_mnist_cifar100_tinyimagenet.pth"

checkpoint = torch.load(checkpoint_path, map_location=device)

model = MultiDatasetProxyVLM(
    vocab_size=checkpoint["vocab_size"],
    embed_dim=checkpoint["embed_dim"],
    text_hidden_dim=checkpoint["text_hidden_dim"],
    pad_id=checkpoint["token_to_id"]["<pad>"]
).to(device)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

class_token_ids = checkpoint["class_token_ids"].to(device)
all_prompts = checkpoint["all_prompts"]
all_class_names = checkpoint["all_class_names"]

print("Model loaded successfully.")
print("Saved epoch:", checkpoint["epoch"])
print("Best saved accuracy:", checkpoint.get("best_total_acc", "N/A"))


# ============================================================
# 3. Test Transforms
# ============================================================

IMG_SIZE = 64

mnist_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5, 0.5, 0.5),
        std=(0.5, 0.5, 0.5)
    )
])

cifar_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5071, 0.4867, 0.4408),
        std=(0.2675, 0.2565, 0.2761)
    )
])

tiny_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4802, 0.4481, 0.3975),
        std=(0.2302, 0.2265, 0.2262)
    )
])


# ============================================================
# 4. Dataset Wrappers
# Label Mapping:
# MNIST: 0-9
# CIFAR-100: 10-109
# TinyImageNet-200: 110-309
# ============================================================

class MNISTTestDataset(Dataset):
    def __init__(self, root="./data"):
        self.dataset = datasets.MNIST(
            root=root,
            train=False,
            download=True,
            transform=mnist_test_transform
        )

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        unified_label = label
        return image, unified_label


class CIFAR100TestDataset(Dataset):
    def __init__(self, root="./data"):
        self.dataset = datasets.CIFAR100(
            root=root,
            train=False,
            download=True,
            transform=cifar_test_transform
        )

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        unified_label = label + 10
        return image, unified_label


class TinyImageNetTestDataset(Dataset):
    def __init__(self, root="./data"):
        val_dir = os.path.join(root, "tiny-imagenet-200", "val")

        if not os.path.exists(val_dir):
            raise FileNotFoundError(
                "TinyImageNet validation folder not found. "
                "Expected: ./data/tiny-imagenet-200/val"
            )

        self.dataset = datasets.ImageFolder(
            root=val_dir,
            transform=tiny_test_transform
        )

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        unified_label = label + 110
        return image, unified_label


# ============================================================
# 5. Create Test Loaders
# ============================================================

batch_size = 128

mnist_test = MNISTTestDataset(root="./data")
cifar100_test = CIFAR100TestDataset(root="./data")
tinyimagenet_test = TinyImageNetTestDataset(root="./data")

mnist_loader = DataLoader(
    mnist_test,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True if device == "cuda" else False
)

cifar100_loader = DataLoader(
    cifar100_test,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True if device == "cuda" else False
)

tinyimagenet_loader = DataLoader(
    tinyimagenet_test,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True if device == "cuda" else False
)


# ============================================================
# 6. Evaluation Function
# ============================================================

def test_model(model, loader, class_token_ids, dataset_name):
    model.eval()

    correct = 0
    total = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc=f"Testing {dataset_name}"):
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images, class_token_ids)
            preds = logits.argmax(dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    acc = 100 * correct / total

    print(f"\n{dataset_name} Test Accuracy: {acc:.2f}%")
    print(f"{dataset_name} Correct / Total: {correct}/{total}")

    return acc, all_preds, all_labels


# ============================================================
# 7. Test Separately
# ============================================================

mnist_acc, mnist_preds, mnist_labels = test_model(
    model,
    mnist_loader,
    class_token_ids,
    "MNIST"
)

cifar100_acc, cifar100_preds, cifar100_labels = test_model(
    model,
    cifar100_loader,
    class_token_ids,
    "CIFAR-100"
)

tinyimagenet_acc, tinyimagenet_preds, tinyimagenet_labels = test_model(
    model,
    tinyimagenet_loader,
    class_token_ids,
    "TinyImageNet-200"
)


# ============================================================
# 8. Summary
# ============================================================

print("\n================ Final Test Results ================")
print(f"MNIST Test Accuracy:          {mnist_acc:.2f}%")
print(f"CIFAR-100 Test Accuracy:      {cifar100_acc:.2f}%")
print(f"TinyImageNet-200 Accuracy:    {tinyimagenet_acc:.2f}%")
print("====================================================")

Using device: cuda


NameError: name 'MultiDatasetProxyVLM' is not defined